In [ ]:
# 8회차 — 입력 해상도 640 -> 960 (v1, 2026-08-04)
# 변수는 imgsz 하나. 나머지는 5회차와 동일 (fire 단일 클래스, 배치 16)
print('round8 notebook v1')
!nvidia-smi -L
!pip -q install ultralytics==8.3.* kagglehub

In [ ]:
# [1] 업로드 — kitchen-fire-poc.zip · assets_1~4.zip (4개면 충분, smoke 소재 불필요)
import zipfile, os, glob
from google.colab import files
up = files.upload()
os.makedirs('/content/work', exist_ok=True)
for n in up:
    zipfile.ZipFile(n).extractall('/content/work')
os.chdir('/content/work')
for d in ('bases', 'flamelib', 'negsrc', 'eval_neg', 'weights'):
    p = f'assets/{d}'
    print(f'{d:10s}', len(glob.glob(p + '/*')) if os.path.isdir(p) else '없음')

In [ ]:
# [2] D-Fire
import kagglehub
DFIRE = kagglehub.dataset_download('sayedgamal99/smoke-fire-detection-yolo')
print(DFIRE)

In [ ]:
# [3] 평가셋 A·C + 학습용 배경 (5회차와 동일 절차)
!python scripts/dfire_eval_set.py --dfire "$DFIRE" --out eval

In [ ]:
# [4] 합성 — 5회차 구성 그대로 (fire 단일 클래스)
!python scripts/synthesize.py --assets assets --out ds \
    --dfire-bg-list eval/train_bg.txt --dfire-bg-count 600 --haze-prob 0.5
!cat ds/data.yaml

In [ ]:
# [5] 학습 — imgsz 960, 배치 16 유지 (약 1시간 20분)
# OOM 이 나면 batch=8 로 낮추고, 사전 등록 문서에 배치 변경을 한계로 기록할 것
!yolo detect train model=yolov8s.pt data=ds/data.yaml epochs=60 imgsz=960 \
    batch=16 project=/content/runs name=r8 exist_ok=False

In [ ]:
# [6] 채점 — 5회차 기준선과 나란히
import glob, os
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)
print('=' * 64); print('[5회차 기준선 · imgsz 640]'); print('=' * 64)
!python scripts/eval_gate.py --weights assets/weights/round5_best.pt \
    --eval-dir eval --cctv assets/eval_neg --conf 0.10
print('\n' + '=' * 64); print('[8회차 · imgsz 960]'); print('=' * 64)
!python scripts/eval_gate.py --weights "$best" --eval-dir eval --cctv assets/eval_neg --conf 0.10

In [ ]:
# [7] 추론 시간 — 실시간 제약 판단용. 같은 이미지 100장으로 두 모델 비교
import glob, os, time
from ultralytics import YOLO
imgs = sorted(glob.glob('assets/eval_neg/*.jpg'))[:100]
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)
for tag, w, sz in (('5회차 640', 'assets/weights/round5_best.pt', 640),
                   ('8회차 960', best, 960)):
    m = YOLO(w)
    m.predict(imgs[:8], imgsz=sz, verbose=False)          # 워밍업
    t0 = time.time()
    for p in imgs:
        m.predict(p, imgsz=sz, verbose=False)
    dt = (time.time() - t0) / len(imgs) * 1000
    print(f'{tag}: 프레임당 {dt:.1f} ms  ({1000/dt:.1f} FPS)')

In [ ]:
# [8] 미탐 사례 비교 — 해상도가 작은 화염을 살렸는지 눈으로 확인
import glob, os, cv2, numpy as np
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
A = [l.strip() for l in open('eval/eval_pos.txt') if l.strip()]
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)
m5, m8 = YOLO('assets/weights/round5_best.pt'), YOLO(best)

def hit(m, p, sz):
    r = m.predict(p, conf=0.10, imgsz=sz, verbose=False)[0]
    return (len(r.boxes) > 0), r

gained = []
for p in A:
    h5, _ = hit(m5, p, 640)
    h8, r8 = hit(m8, p, 960)
    if h8 and not h5:
        gained.append((p, r8))
print(f'5회차가 놓치고 8회차가 잡은 것: {len(gained)}장')
tiles = []
for p, r in gained[:8]:
    im = cv2.imread(p)
    for b in r.boxes.xyxy.cpu().numpy().astype(int):
        cv2.rectangle(im, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 2)
    tiles.append(cv2.resize(im, (400, 225)))
if tiles:
    while len(tiles) % 4: tiles.append(np.zeros((225, 400, 3), np.uint8))
    cv2_imshow(np.vstack([np.hstack(tiles[i:i+4]) for i in range(0, len(tiles), 4)]))

In [ ]:
# [9] 가중치 내려받기
import glob, os
from google.colab import files
files.download(max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime))